# NeuroScale++ — ANN-to-SNN Adaptive Inference on Google Colab

This notebook demonstrates the **NeuroScale++** pipeline for energy-efficient ANN-to-SNN adaptive inference using the modular package structure.

**Full pipeline (~15–20 min on T4 GPU with default fast settings):**
1. **Environment Setup** — Clone repository (if on Colab) and install requirements
2. **Phase 1** — Pretrain ANN (ResNet-20) on CIFAR-10
3. **Phase 2** — Convert ANN→SNN, profile per-sample scaling laws, fit (α, β, γ)
4. **Phase 3** — Joint train: Complexity Predictor + Multi-Exit SNN branches
5. **Evaluation** — Adaptive inference, energy savings, plots

> ⚡ **Before running:** `Runtime → Change runtime type → GPU (T4)`

In [ ]:
# ── Setup and Install ─────────────────────────────────────────────────────────
import os
import sys
from pathlib import Path

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    print("Running on Google Colab. Cloning repository...")
    !git clone https://github.com/Vighneshwarkuru/NeuroScale.git
    %cd NeuroScale
    !pip install -r requirements.txt
    sys.path.append(os.getcwd())
else:
    print("Running locally. Make sure you are in the repository root.")
    if not Path('neuroscale').exists():
        print("Warning: 'neuroscale' directory not found. Please run this notebook from the repo root.")

import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

torch.manual_seed(42); np.random.seed(42)
print("Setup complete!")

## Configuration
We load the configuration from `configs/cifar10.yaml` and adjust training parameters for a faster interactive run.

In [ ]:
from neuroscale.utils.config import load_config

# Load YAML configuration
config = load_config("configs/cifar10.yaml")

# Tune config parameters for a fast demo run (~15 mins total)
config['model']['epochs'] = 15          # Standard: 30-100
config['profiling']['num_samples'] = 2000 # Standard: 5000-20000
config['predictor']['epochs'] = 10       # Standard: 20-30

print("Config loaded successfully!")
print(f"ANN Epochs: {config['model']['epochs']}")
print(f"Profiling Samples: {config['profiling']['num_samples']}")
print(f"Predictor Epochs: {config['predictor']['epochs']}")

## Phase 1 — ANN Pretraining
Train a ResNet-20 on CIFAR-10. This model acts as our pretrained backbone.

In [ ]:
from neuroscale.training.phase1_ann import train_ann
from neuroscale.utils.results_manager import ResultsManager

rm = ResultsManager(base_dir='./results', config=config)

print("Starting Phase 1 (ANN Pretraining)...")
ann_model = train_ann(
    config=config,
    device=device,
    checkpoint_dir='./checkpoints',
    log_dir='./logs',
    results_manager=rm
)
print("Phase 1 complete!")

## Phase 2 — SNN Conversion & Profiling
Convert the trained ANN to an SNN using threshold balancing, run the training subset through the SNN to measure temporal output accuracy, and fit power-law scaling curves per sample.

In [ ]:
from neuroscale.datasets.factory import get_dataloaders
from neuroscale.training.phase2_profiling import profile_snn

# We need the training data loader for profiling
train_loader, _ = get_dataloaders(config, data_dir='./data')

print("Starting Phase 2 (SNN Conversion & Profiling)...")
profiling_results = profile_snn(
    ann_model=ann_model,
    train_loader=train_loader,
    config=config,
    device=device,
    save_dir='./checkpoints',
    log_dir='./logs',
    results_manager=rm
)
print("Phase 2 complete!")

## Phase 3 — Joint Training
Train the **Complexity Predictor** (image to scaling law parameters α, β, γ) and the **Multi-Exit SNN** exit branches.

In [ ]:
from neuroscale.training.phase3_joint import train_joint
from neuroscale.spiking.snn_model import SNNModel
from neuroscale.conversion.converter import ANNtoSNNConverter

# Re-load checkpoints / thresholds for building the SNN wrapper
snn_ckpt = torch.load('./checkpoints/snn_converted.pth', map_location=device)
thresholds = snn_ckpt['thresholds']

# Reconstruct normalized model base
calib_loader, _ = get_dataloaders(config, data_dir='./data')
converter = ANNtoSNNConverter(percentile=config['conversion']['percentile'], calibrate=True)
snn_base, _ = converter.convert(ann_model, calib_loader, device)

snn_model = SNNModel(
    ann_model=snn_base,
    thresholds=thresholds,
    max_timestep=config['snn']['max_timestep'],
    neuron_type="if"
).to(device)
snn_model.load_state_dict(snn_ckpt['snn_state_dict'])

print("Starting Phase 3 (Joint Training)...")
me_snn, predictor = train_joint(
    snn_model=snn_model,
    train_loader=train_loader,
    profiling_results=profiling_results,
    config=config,
    device=device,
    checkpoint_dir='./checkpoints',
    log_dir='./logs',
    results_manager=rm
)
print("Phase 3 complete!")

## Evaluation — Adaptive Inference
Evaluate accuracy, average timesteps, and energy savings on the test set.

In [ ]:
from neuroscale.inference.adaptive_inference import AdaptiveInference

# Create data loaders for testing
_, test_loader = get_dataloaders(config, data_dir='./data')

print("Running Adaptive Inference Evaluation...")
evaluator = AdaptiveInference(
    multi_exit_snn=me_snn,
    complexity_predictor=predictor,
    config=config,
    device=device
)

results = evaluator.evaluate(test_loader)

print(f"\n=== Evaluation Results ===")
print(f"ANN Baseline Accuracy : {results['ann_baseline_accuracy']:.2f}%")
print(f"NeuroScale++ Accuracy : {results['adaptive_accuracy']:.2f}%")
print(f"Average Timesteps Used: {results['average_timesteps']:.2f} / {config['snn']['max_timestep']}")
print(f"Energy Savings        : {results['energy_savings']:.2f}%")
print(f"Speedup Factor        : {results['speedup']:.2f}x")

## Visualizing Results
Plot the accuracy vs latency curves, parameter distributions, and adaptive inference metrics.

In [ ]:
from neuroscale.utils.plotting import generate_all_plots

print("Generating analysis plots...")
# Generate all plots using the results saved by ResultsManager
generate_all_plots(results_dir=rm.results_dir)

# Display some of the generated plots
from IPython.display import Image, display
plot_dir = Path(rm.results_dir) / 'plots'

for plot_file in plot_dir.glob('*.png'):
    print(f"\nPlot: {plot_file.name}")
    display(Image(filename=str(plot_file)))